In [ ]:
# Run this first to get the correct styling for the Jupyter Notebook
# This cell is nothing to do with the simulation, it's entirely to do with the Jupyter Notebook formatting
from IPython.core.display import HTML
import urllib.request
def css_styling():
    with urllib.request.urlopen("https://raw.githubusercontent.com/uolphysicsteaching/resources/main/notebook.css") as response:
        return HTML(response.read().decode("utf-8"))

try:
    css_styling()
except:
    print("Unable to import formatting data. Likely an internet connectivity issue :( )")

# Molecular Dynamics Simulator

<div class='summary'>

Welcome to the __PHYS3190 Molecular Simulation: Molecular Dynamics Simulator__. This is a piece of code written by the module team for you to study and use. Feel free to download and edit a copy, as there will always be a working version available on Minerva :)
    
### Learning Aims
The aim of this course is to provide you, the student, with the fundamental knowledge of how molecular simulations work in general. We will approach this via the study (and simulation of) of a specific system: the Lennard-Jones gas. We will study various aspects of the Lennard-Jones gas, and build up to a working simulation one step at a time so that you understand each step of the process. From there, you will understand the theoretical underpinnings of this system and, with any luck, the general method, such that you could edit this simulation suite to study systems other than the Lennard-Jones gas by adding your own forces and interactions between the particles! 

### Course Learning Objectives

The objectives of this part of the course are:
    
- To understand the theoretical model of the Lennard-Jones gas
- To understand the computational implementation of a simulation of the Lennard-Jones gas
- To understand the general method of performing dynamic molecular simulations using both Brownian dynamics, and maybe Langevin dynamics

### Course Learning Outcomes
    
At the end of this session you will be able to:
    
- Describe the structural components and force-fields that comprise a Lennard-Jones gas 
- Calculate single particle properties within the Lennard-Jones gas (Diffusion constant, kinetic energy etc)
- Calculate ensemble particle properties of the Lennard-Jones gas (expected kinetic energy, Van der Waals interaction energies, average diffusion constants, pressures etc)
    
</div>

## How this works
The following notebook has many code blocks. Each should be executed in order to perform a simulation. If you need to make a change to any of the code, it would be best practice edit the specific code and re-run all cells after that. When you gain more understanding of what the program is doing, you may begin to re-run specific cells as needed.

When a concept appears which may be new to you, I will use a markdown block (one of these fancy, formatted text blocks) to explain that concept. Some will be computational concepts, other will be physical concepts. I will clarify which it is in the block :). Once you have learned the concept, you can delete markdown blocks if you want. They do not affect the code in any way. Even if you understand the topic, have a read. You may learn something new and interesting...

The code is also well commented. Comments are there for two reasons. First, for the programmer (i.e. me!) to remember what the code they wrote did. Second, for anyone else reading the code (i.e. you!) to understand what it does. In other words, read the comments! And write your own if you make any changes to the code

Finally, there are also a few little philosophical ponderings throughout the notes (usually written as footnotes in each section). Read them if you want, but they're not examinable! Just a little something to think about :)

## What is a Lennard-Jones Gas?
When you begin studies of thermodynamics, even pre-University, you come across the idea of an ideal gas. The "ideal" part of the ideal gas is that to derive the equation describing it's macroscopic behaviour ($PV=nRT$), you have to make the assumption that none of the particles are interacting with one another. This is absolutely not true in general! However, calculating the behaviour of a system of interacting particles is really hard...and so we use computer simulations to help us do the calculations.

The most basic interaction type we can introduce to begin building a more realistic model of a gas are Van der Waals interactions. These can be modelled quite well by the Lennard-Jones 6-12 potential:

$$
U(r) = \epsilon \left( \left( \frac{r_0}{r} \right)^{12} - 2\left( \frac{r_0}{r}\right)^6 \right)
$$

and that is precisely what this code can do. By choosing the appropriate material parameters, simulation parameters and types of force acting on and between each particle, we can begin to simulation a realistic gas. Use this code to do that :)

<hr class='double' />

## Full Code Begins

<hr class='double' />

In [ ]:
#
# Module imports
#

# Importing external modules that we'll need
import sys, os
import numpy as np
from matplotlib import pyplot as plt

# Importing the classes we have defined
from modules.SystemParameters import PhysicalConstants
from modules.SystemParameters import SimulationParameters
from modules.PhysicalSystem import PhysicalSystem
from modules.SimulationBox import SimulationBox
from modules.NumericalIntegrator import NumericalIntegrator
from modules.FileIO import FileWriter
from modules.SimulationRunner import SimulationRunner
from modules.SimulationTester import SimulationTester

<hr class='double' />

### Computational Concept: Importing Python Objects

You've no doubt used the `import` statement before to get access to things like `numpy` and `matplotlib`. You may even recognise statements such as `from numpy import pi`, which would allow us to use the value of `pi` directly, without having to say `np.pi` or `numpy.pi`. What, then, is `from modules.SystemParameters import PhysicalConstants`?

In the main code directory, you will see a folder called `modules/`. This is the `modules.` part of each of the above import statements. Inside the `modules/` folder, you will see a file called "SystemParameters.py". Open it up in Jupyter and have a look at its contents. You'll notice that inside, there is a `class` called `PhysicalConstants`. Thus, the line `from modules.SystemParameters import PhysicalConstants` means that we need to import something called `PhysicalConstants` from a file called `SystemParameters.py` in a folder called `modules/`. We can also see another `class` in that file called `SimulationParameters`. Is that also imported in this simulation script?

When we import something in Python, the Python interpreter finds the appropriate file and object on your computer and copies all of the code into your script at that point. This means we can write our own code, put it in another file, and import it! So long as Python knows where to look to find the code being imported, it's all good! In this case, Python knows where the code is because it's in the same directory as this script. You can look into it in more detail if you want using the following link, but it's not necessary :) https://docs.python.org/3/reference/import.html


### Computational Concept: Python Classes

Ok, but what is this thing called a `class`? Well, Python is an "object-oriented language", meaning that data structures in Python can take the form of user-defined containers called classes. Take the class `PhysicalConstants` for example. Once imported, I can write a line of code as follows:

`myConstants = PhysicalConstants()`

The variable `myConstants` is now of data type `PhysicalConstants`. Just like if you were to make an `int`, a `float`, a `str`, we have made a `PhysicalConstants` type object. So what is a `PhysicalConstants`? Well, we can see by looking at the code of the `PhysicalConstants` class (in SystemParameters.py) that it defines a variable, `self.kb`. The sharp-eyed among you might realise that this is Boltzmann's Constant with non-standard units, but what this means computationally is that I can access the variable `kb` as follows:

`print(myConstants.kb)`

`myConstants` is of type `PhysicalConstants`, and `PhysicalConstants` *contains* the variable `kb`. Therefore, I can access it with the dot notation. You may recognise this from when you have previously accessed the value of `pi` using `np.pi`...which means that `numpy` is a file somewhere on your computer that contains that value of pi! Amazing, we now understand something very deep about the structure of the Python language<sup>1</sup> :)

But numpy has other things, doesn't it? It has functions attached to it, like `np.cos` that takes a parameter. Does `PhysicalConstants` have anything like this? Well, it has this `__init__` function, but as you may have guessed, this just initialises the object. More interestingly, have a look at the function `applyThermalForces` in "PhysicalSystem.py". Functions attached to classes are called "methods" and the only major difference between regular functions and methods is that the first parameter in a method should be the keyword `self`. This allows the method to access the variables stored on the class using `self.` i.e. `self.kb` in the `PhysicalConstants` class

<div class='summary'>
    
**SUMMARY:** A class is a data-structure that contains variables and methods which can be accessed via the dot operator. Each new class object created is entirely independent and can be changed independently of all other objects of the same type...just like regular variables! When using the dot notation to access variables and methods, those variables and methods come from that specific object. Hence, if I have `a` and `b` that are both of type `PhysicalConstants`, `a.kb` and `b.kb` will begin with the same value of `kb`, but if you change the value `a.kb`, `b.kb` will not be changed.
    
**NOTE:** Classes and data-structures are fascinating and are a level of abstraction that can change how you view programming generally. This is nothing to worry about, but please don't forget that everything you've learned so far is still true! It's just...a little more complex than you thought it was, under the hood.
</div>

<sup>1</sup>Traditional spoken/written languages also work this way. Consider the word "chair". We all know what a chair is, but...why? It's because in language, a word is a wrapper that contains within it many underlying concepts that define that word. To me, the word "chair" implies something that has legs and that you can sit on. So, I might have a __class__ called `Chair` and say `myChair = Chair()`. Then, I may decide to say that `myChair.numLegs = 4` or something like that, and have a method called `myChair.sit(a)`. But the fact that `Chair` has a variable attached called `numLegs` and a function called `sit()` implies that the class `Chair` is *defined* by having legs and by being able to be sat on...the same as in traditional languages. You may disagree with my definition of "chair", and this is entirely fine. We can discuss, and redefine the class as appropriate...which is exactly what is happening right now with certain words and concepts in language :) if you're interested in the philosophy underlying computer programming, maths, language, and how it all fits together, come chat to me. I could talk about it all day! Here are a few wikipedia links:

https://en.wikipedia.org/wiki/The_Principles_of_Mathematics

https://en.wikipedia.org/wiki/Philosophical_Investigations

<hr class='double' />


In [3]:
#
# Functions defined here (for future use)
#

# Intro stuff
def printIntro():
    
    print("##########################################")
    print("###### Molecular Dynamics Simulator ######")
    print("##########################################")
    print("")

def printSimulationInformation(params):
    print("Simulation Data:\n")
    print("Number of particles: %d" % (params.numParticles))
    print("Volume fraction: %.3f" % (params.volumeFraction))
    print("Temperature: %.2f K" % (params.temperature))
    print("Solvent viscosity: %.2f MPa.ns" % (params.viscosity))
    print("Particle radius: %.2f nm" % (params.particleRadius))
    print("Particle drag: %.2f pN.nm/ns" % (params.drag))
    print("Particle density: %.2f kDa/nm^3" % (params.particleDensity / ((5.0/3.0)*1e-3)))   # Converstion from internal units to user units
    print("Particle mass: %.2f kDa" % (params.particleMass / ((5.0/3.0)*1e-3)))      # Converstion from internal units to user units
    print("kT: %.2f pN.nm" % (params.kbt))
    
    print("")
    print("Timestep: %.2e ns" % (params.dt))
    print("Simulation Time: %.2e ns" % (params.simulationTime))
    print("Frame Rate: %.2e ns" % (params.frameRate))
    print("Number of Steps: %d steps" % (params.numSteps))
    print("Output Rate: %d steps" % (params.outputRate))
    
    print("")
    print("Boundary conditions: %s" % (params.boundaryConditions))
    print("LJ Energy minimum: %.2f pN.nm" % (params.LJeps))
    print("LJ Equilibrium Distance: %.2f nm" % (params.LJr0))
    print("LJ Cutoff Distance: %.2f nm" % (params.LJcutoff))
    
    
    print("")
    print("Equilibration Trajectory Filename: %s" % (params.tFname_Equilibration))
    print("Equilibration Measurement Filename: %s" % (params.mFname_Equilibration))
    print("Production Trajectory Filename: %s" % (params.tFname_Production))
    print("Production Measurement Filename: %s" % (params.mFname_Production))

def printBoxInformation(box):
    print("Simulation Box Data:\n")
    print("Number of voxels: %d (%d,%d,%d)" % (box.totalNumVoxels, box.numVoxels[0], box.numVoxels[1], box.numVoxels[2]))
    print("Dimension: %d,%d,%d" % (box.dimension[0], box.dimension[1], box.dimension[2]))
    print("Voxel Dimension: %d,%d,%d" % (box.resolution, box.resolution, box.resolution))

    print("Boundary Conditions: %s" % (box.bc))

# Outro stuff
def printOutro():
    
    print("##########################################")
    print("######### Thanks for playing! :) #########")
    print("##########################################")
    print("")

---

## Main Program Flow Begins

---

In [ ]:
# Print the simulation information to screen
printIntro()

In [ ]:
# Get some physical constants by instantiating the container class
pConst = PhysicalConstants()

In [ ]:
# Set some parameters (change these to change the physical behaviour of the simulations)
params = SimulationParameters()

# Physical parameters (change these to change the physics of the simulations)
params.simulationType = "Brownian"
params.numParticles = 100   # Number of particles to include in the simulation
params.volumeFraction = 0.01
params.temperature = 150.0   # K (298=room temp)
params.viscosity = 1   # MPa.ns (1 = water)
params.setDensity(1)  # kDa/nm^3 (1 = approximate average protein density, Fischer et al, Protein Sci. 2004 Oct; 13(10): 2825–2828)
                        # There a setter method here for a good reason. Check it out!
    
params.particleRadius = 0.5 # nm

In [ ]:
# Simulation parameters (change these to change the computational behaviour of the simulations)
params.dt = 0.01   # ns (the success and accuracy of your simulation is dependent on this value!)
params.simulationTime = 100 # ns (total amount of simulation time to run)
params.frameRate = 1  # ns (our simulation will output data every "frameRate" frames)

# Box parameters
params.boundaryConditions = ""  # Boundary condition type (hbc:hard boundary conditions)
params.LJeps = 0.0  # Energy minimum for Lennard-Jones interactions
params.LJr0 = 0.0  # Energy minimum for Lennard-Jones interactions
params.LJcutoff = 3.0

params.mFname_Equilibration = ""  # Filename used to write (equilibration) measurement data
params.tFname_Equilibration = ""  # Filename used to write (equilibration) trajectory data
params.mFname_Production = ""  # Filename used to write (production) measurement data
params.tFname_Production = ""  # Filename used to write (production) trajectory data

In [ ]:
# Parameter validation
if not params.validate():
    sys.exit("Inconsistent parameters. Please fix and try again!")

In [ ]:
# Now get all of the "non-primitve" values, calculated from the parameters given
params.calculateNonPrimitives(pConst)

In [ ]:
# Print some more information, so we know we have the right stuff
# If this is wrong, go back and alter the values!
printSimulationInformation(params)

In [ ]:
#
# Build the simulation system
#

# Physical system
try:
    pSys = PhysicalSystem(params.numParticles)
    
except Exception as e:
    print("Error:", e)
    print("Error: Unable to create SimulationBox :(")

<hr class='double' />

### Simulation Concept: Physical System

The object here called `PhysicalSystem` is the name I have given to the object which contains all of the position and velocity information about the system i.e it contains intrinsic information about the particles. Thus, this object contains lists of all of the position, velocity and force vectors for the system. If we have knowledge of all positions, velocities and forces (i.e. accelerations), then from statistical mechanics, we can (in principle) calculate everything there is to know about this system (internal stresses, pressures, energies, free energies, pressures etc).

<div class='summary'>
    
**SUMMARY:** The Physical System is simply a data structure, containing all of the physical information we need
</div>

<hr class='double' />

In [ ]:
# Computational box
particleVolume = 4.0/3.0 * np.pi * params.particleRadius**3
boxLength = (params.numParticles * particleVolume / params.volumeFraction) ** (1.0/3.0) # Given a volume fraction, how big does the box need to be for N particles?


resolution = params.particleRadius * 10.0 # Should be much bigger than a single particle (+ LJ radius), but nowhere near the size of the box
                                            # There are cleverer ways to calculate this based on the forces involved...see if you can think of one?
# Try to create the box
try:
    box = SimulationBox(params.boundaryConditions, length=boxLength, resolution=resolution)
    
except Exception as e:
    print("Error:", e)
    print("Error: Unable to create SimulationBox :(")

# Info for user
printBoxInformation(box)

<hr class='double' />

### Simulation Concept: Computational Box

A computational box exists for two reasons, one of which is kind of physical, and one of which definitely isn't. Nevertheless, both reasons are absolutely necessary to perform efficient simulations.

1. Introduce physical boundary conditions to the simulation (mostly physical)
    - In real physical systems, it is hardly ever the case that a system exists in complete isolation. In a fluid, for example, the existence of fluid channels with friction (such as river banks, blood vessel walls (veins, arteries etc)) changes the physical behaviour of the fluid so that its velocity profile is not uniform. This vastly changes the behaviour of things that interact with the fluid as well as the fluid itself. Thus, we can introduce a container to control alter the physics of the system we are using. For a Lennard-Jones gas, we can introduce a box that contains all of the particles in order to keep its volume constant. Thus, when calculating the thermodynamics of the whole thing, we can be sure that we are taking the correct partial derivatives (for example, we can calculate the entropy from the Helmholtz free energy if we hold the volume constant in the canonical ensemble: https://en.wikipedia.org/wiki/Helmholtz_free_energy)
    
    - However, we often want to simulate large, almost infinite systems so that we can measure bulk properties in the absence of of boundary effects. In this case, we cannot just increase the number of particles to macroscopic scales (moles/ Avagadro's number of particles). Even the best computers on Earth cannot handle the amount of calculations required to do that (some researchers have tried to simulate entire biological cells and they have to make very non-physical approximations such as "none of the proteins interact with one another"). To make representations of infinite systems while at the same time limiting the number of required particles, we introduce periodic boundary conditions. What this means is that if a particle moves through one side of the box, rather than interacting with the box itself, it passes through and comes back in the other side. Equally, particles can interact with one another across the boundary. In practice, so long as particles do not have long-range interactions with one another (so, for example, they cannot interact with themselves!), this will accurately represent an infinite system. The Van der Waals potental (i.e. the Lennard-Jones equation) is very short range so we are ok, but electrostatics generally? They are long range. Hydrodynamic interactions are also long range, and so if we had included these, more care would be needed to implement periodic boundary conditions. But for now, we're ok :)
    
2. Reduce the number of required calculations in order to speed up the simulation (very non-physical)
    - Following on from above, the Lennard-Jones interaction is short range. For large distances (relative to the LJ minimum distance) the strength of forces between particles would be negligible, so wouldn't it be nice if we could just...ignore them? That's a classic physics technique right there: ignore the irrelevant<sup>2</sup> forces to make the maths easier! Well that would be nice, but in a simulation it seems like that for every possible pair of particles, we would first have to calculate the distance between particles (a reasonbly costly thing potentially involving a square-root) and then check whether it is less than some pre-defined distance before doing a calculation. Wouldn't it be even nicer if we didn't even have to calculate that distance in the first place?
    
    - That's where the box comes in! We divide the box into voxels (like 3D pixels) and, at the beginning of each simulation step, we assign each particle to the voxel it contains. Then, rather than looping over all pairs of particles, we instead loop over every pair of *adjacent* voxels! This requires much less computational effort, reducing the calculation from being O($N^2$) (i.e. the calculation scales with number of particles squared) to O($N$ln$\left(N\right)$, which is much more efficient. This is a purely computational thing and has nothing to do with the physics. However, if we do it correctly, and make sure the size of each voxel is significantly bigger than the LJ interaction minimum distance, it will not affect the calculations in a percievable way :)
    
<div class='summary'>
    
**SUMMARY:** A computational box allows us to apply physical boundary conditions to our simulation, and increase the efficiency of the computation itself by making some clever data structures in advance
    
</div>

<sup>2</sup>Determining whether or not a force is truly irrelevant is difficult, perhaps impossible. Perhaps you've heard of the butterfly effect? The force I choose to ignore may turn out to have a huge, chaotic effect on the system down the line that I simply can't predict. We do our best to understand and simulate the world around us, but there's no way to be absolutely 100% certain that something is irrelevant. Still, if it's a weak, short-range force, then ignoring long-term effects is a good first approximation :)
<hr class='double' />

In [ ]:
# Build a numerical integrator object
try:
    numericalIntegrator = NumericalIntegrator(params.simulationType)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to create NumericalIntegrator :(")

<hr class='double' />

### Simulation Concept: Numerical Integrator

The entire reason we have to do computer simulations in the first place is because the equations of motion for complex systems are literally impossible to solve using currently available analytical methods. Most real systems (gases, liquids, solids, biological, engineered stuff, buildings, planes, the atmosphere, stars, galaxies, human society, the economy) are all way too complex to solve, and so require computer simulations.

When we solve an equation of motion, what we are generally doing is trying to find out how the system will change over time. Sometimes we care about temperature over time, energy over time etc. In the case of the LJ Gas simulation, we are finding how the positions of many particles change over time. This means we will need to do numerical integration where our small step $\Delta h$ is a small step forwards in time, $\Delta t$.

Check out my lecture notes on this, but generally, the same rules apply to this as to what you learned if you previously did numerical integration (trapezium rule and all that stuff). There are certain rules on how big $\Delta t$ can be. If it's too big, the simulation will go "numerically unstable" and fail. It is usally clear when this happens as the energies involved start increasing over time, tending towards infinity<sup>3</sup>. Generally speaking, the larger the forces are in your system, the smaller $\Delta t$ needs to be to be able to "resolve" the equations of motion. Forces change velocities, and velocities are associated with energies. Thus, we need to keep our velocities accurate by looking at suitable small changes in forces.

Over the course of the module you'll learn that when a Hookean spring is involved, then for a Brownian dynamics simulation, the timestep needs to be such that $\Delta t < \lambda / k$, where $\lambda$ is the drag, and $k$ is a spring constant. We don't have any springs, but we do have LJ interactions. We can associate a spring constant with the LJ interaction by taking the second derivative of the energy function and evaluting at $r_0$...

In a Langevin simulation, there is a different condition that must be satisfied, $\Delta t < m / \lambda$, where $m$ is the mass of a particle. Your $\Delta t$ must be smaller than the smallest timescale in the system! But these conditions are just the absolute upper limits. Ideally, $\Delta t$ should be as small as possible while still running your simulation in a reasonable time. The smaller it is, the more accurate the numerical calculations will be. I usually go with 10x smaller than the smallest timescale in the simulation.

<div class='summary'>
    
**SUMMARY:** Numerical integrators update the positions and velocites of your PhysicalSystem. The timestep they use, $\Delta t$, must be smaller than the smallest timescale in your system. For a Brownian simulation, $\Delta t < \lambda / k$. For a Langevin simulation, $\Delta t < \lambda / k$ and $\Delta t < m / \lambda$. Ideally, you should make your timescales as small as you can while still running your simulation in a reasonable time. The smaller it is, the more accurate the numerical calculations will be.
</div>

<sup>3</sup>You know hyper-inflation, where the apparent value of money goes wildly out of control with respect to the actual value of goods? Well, this occurs because economies are large systems of humans (particles) interacting with one another and exchanging money (energy). If the changes in rates of value or external pressures (economic forces) get too large with respect to the time it takes humans to make economic decisions ($\Delta t$), the economy goes numerically unstable. This is a very real-world example of numerical integration going unstable. Perhaps you'll notice that, historically, when this happens the solution has always been to restart the economy. Wipe the debt, create new money and new exchange rules. Effectively, they are turning it off and on again xD and we should do the same with our simulations! If you're interested in this sort of thing (the statistical mechanics of the economy), I wrote an essay on it a few years back: https://abstractacademic.co.uk/politics-economics/fairness-at-scale
<hr class='double' />

In [ ]:
# Get the output files open and ready for writing the equilibration files
# I'll explain what equilibration is shortly!
try:
    fw_Equilibration = FileWriter(params.tFname_Equilibration, params.mFname_Equilibration)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to create FileWriter for equilibration :(")
    
try:
    fw_Equilibration.openOutputFiles(params)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to open the equilibration output files for writing :(")

In [ ]:
# Get the output files open and ready for writing the production files
# I'll explain what production is shortly!
try:
    fw_Production = FileWriter(params.tFname_Production, params.mFname_Production)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to create FileWriter for production :(")
    
try:
    fw_Production.openOutputFiles(params)
except Exception as e:
    print("Error: ", e)
    print("Error: Unable to open the production output files for writing :(")

<hr class='double' />

### Simulation Concept: Output Files

Every simulation protocol has output files for storing the data our simulation produces. We have two: 

- __The trajectory file (.mstraj)__
    - The most basic is a trajectory file, which contains the positions, velocities and forces on every particle at certain times during the simulation. As mentioned earlier, with this information, we can calculate anything we like i.e. energies, pressures, whatever. Even more crucially, we can restart a simulation using this information! We load it in as the new initial state, and simply continue from that point<sup>4</sup>. I haven't implemented that in this script, but for really big simulations that take weeks, it's very important to have this ability (in case the computer breaks, power cuts etc).
- __Measurement file  (.msmeas)__
    - This is not strictly necessary as it can all be calculated from the trajectory itself. However! Things like interaction energies, kinetic energies, centers of mass, are so ubiquitous that I have made the program calculate them for you. But it will be up to you to determine whether your simulation is correct, and what it is telling you about the physics, based on these measurements...

<sup>4</sup>This is just like saving a file from something like MS Word and opening it again later. For the video gamers among you, this is what happens when you save a game. It saves everything it needs to restart at exactly the same place in the program / game you were when you last played.
<hr class='double' />

In [ ]:
#
# Initialise the system
#

# All this will do is randomly distribute the particles throughout the box, ready for initialisation
box.randomInitialise(pSys, params)

In [ ]:
#
# Initialise an object that can perform the actual simulation algorithms
#
runner = SimulationRunner()

In [ ]:
#
# Initialise an object that can test the state of a simulation while it is running
#
tester = SimulationTester()

<hr class='double' />

### Simulation Concept: Simulation Runner / Tester

The object here called `SimulationRunner` is the name I have given to the object which contains all of the methods that perform simulations. It is simply a container, separating the algorithms from the main code flow here for readability.

`SimulationTester` is similar, containing methods that test the progress of the simulation.
<hr class='double' />

In [ ]:
#
# Equilibration begins
#

# Add or remove equilibration stages as needed below!

# Think about the equilibration steps you need to perform the simulations you want

In [ ]:
# Calculation for equilibrationTime goes here
equilibrationTime = 1

# Stage 1: Thermal only
runner.runThermalEquilibration(pSys, params, box, fw_Equilibration, equilibrationTime, numericalIntegrator)

In [ ]:
# Test Stage 1 equilibration
tester.testThermalEquilibration(pSys,params)

<hr class='double' />

### Simulation Concept: Equilibration

Simulations are intended to be representations of reality, yet we have built the system computationally. Before we begin the "production simulation", which can be analysed for physical realism, we must first bring the simulation to an initial state which represents reality accurately. In our case, this means:

- No overlapping particles
- No unrealistically huge forces
- Statistical equilibrium as defined by the canonical ensemble

From a statistical mechanics perspective, a Molecular Dynamics simulation explores various microstates relating to the macrostate defined by our input parameters (temperature, particle mass etc). Thus, before our simulation begins, the system state must correspond to one of those microstates! This is the essence of the equilibration process. 

There are many ways to computationally perform the equilibration process; however, so that we can understand exactly what it is our simulations are doing, we will equilibrate by directly running several different simulations in advance. For the Lennard-Jones gas, for example, we would do the following:

- First, we run a simulation in which the only forces acting are stochastic thermal forces. This will have the effect of generating something like an ideal gas (with a Boltzmann distribution of velocities if we are using the Langevin integration approach). From a statistical mechanics perspective, this will move us into the microcanonical ensemble (no interactions between particles, all microstates equally likely etc).
- Second, we will run a simulation which introduces LJ interactions in the absence of thermal forces. This will continuously relax the system into the LJ minimum without the numerical instability problems that thermal noise would bring. From a statistical mechanics perspective, this will move us into an energy minimum, but not a free energy minumum (as we have turned off temperature)
- Third, we will run a short simulation which introduces all interactions. From a statistical mechanics perspective, this will move us into the canonical ensemble.

If you were to include your own forces in addition to this, you would add them one at a time in reverse order from the forces which have the potential to generate the largest forces to the smallest forces in reverse order, followed by a run with thermal noise at the very end. This will stabilise your simulation as well as possible as it transitions into the canonical ensemble.

<div class='summary'>

**Summary:** Equilibration is required before performing simulations which can be analysed for physical realism. How we equilibrate depends on what interactions we include in the simulation. The goal in all cases, however, is to bring our system into a representative microstate from the ensemble that we are interested in. In our case, this is the canonical ensemble.
    
**Note:** Another way to equilibrate is to use numerical minimisation techniques (Newton-Raphson, gradient decent etc) to find the most stable state for each stage of the simulation. However, running the simulation itself will do this (albeit less efficiently), and it works better for teaching. So that's what we're doing!
</div>

<hr class='double' />

In [ ]:
#
# Production simulation begins
#

# Calculation for productionTime goes here
productionTime = params.simulationTime
fr = params.frameRate

# Final stage: Production simulation
runner.runProduction(pSys, params, box, fw_Production, productionTime, numericalIntegrator, thermal=True, lj=False, frameRate = fr)

In [ ]:
# Simulation has finished. Clean everything up

# Close simulation files
fw_Equilibration.closeOutputFiles()
fw_Production.closeOutputFiles()

# Delete all objects
del tester
del runner
del fw_Production
del fw_Equilibration
del numericalIntegrator
del box
del pSys
del params

# Print Outro
printOutro()